In [1]:
# pip install tiktoken matplotlib

In [2]:
import math
import torch
import tiktoken
from datasets.preprocess import download_the_verdict
from datasets.dataloader import verdictDataLoader
from models.gpt2 import GPT
from configs.model import ModelConfig
from configs.training import TrainingConfig
from evaluation.loss import calc_loss_accuracy_loader,cross_entropy_loss,token_accuracy
from generation.sample_text import generate_sample_text
from configs.scheduler import SchedulerConfig
from configs.optimizer import OptimizerConfig
from configs.checkpoint import CheckpointConfig
from trainer.trainer import Trainer

c:\Users\admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
tokenizer = tiktoken.get_encoding("gpt2")

In [4]:
GPT2_SMALL = ModelConfig(
    emb_dim=32,
    n_layers=2,
    n_heads=4,
    activation="gelu",
    context_length=24
)

TRAIN_CONFIG = TrainingConfig(
    epoch=5,
    batch_size=2,
    stride=24,
    context_length=24
)

OPTIMIZER_CONFIG = OptimizerConfig()
SCHEDULER_CONFIG = SchedulerConfig()
CHECKPOINT_CONFIG = CheckpointConfig()


In [5]:
raw_text = download_the_verdict()
train_data = 0.9  ## train data to be 90%
len_train_data = math.floor(len(raw_text)* train_data)
train_data = raw_text[:len_train_data]
val_data = raw_text[len_train_data:] 


train_dataloader = verdictDataLoader(train_data, TRAIN_CONFIG.batch_size,TRAIN_CONFIG.context_length, TRAIN_CONFIG.stride)
val_dataloader = verdictDataLoader(val_data,TRAIN_CONFIG.batch_size,TRAIN_CONFIG.context_length, TRAIN_CONFIG.stride)

In [6]:
dataiter = iter(train_dataloader)
input_sample, target_sample = next(dataiter)
print(input_sample, target_sample)

tensor([[   40,   367,  2885,  1464,  1807,  3619,   402,   271, 10899,  2138,
           257,  7026, 15632,   438,  2016,   257,   922,  5891,  1576,   438,
           568,   340,   373,   645],
        [ 1049,  5975,   284,   502,   284,  3285,   326,    11,   287,   262,
          6001,   286,   465, 13476,    11,   339,   550,  5710,   465, 12036,
            11,  6405,   257,  5527]]) tensor([[  367,  2885,  1464,  1807,  3619,   402,   271, 10899,  2138,   257,
          7026, 15632,   438,  2016,   257,   922,  5891,  1576,   438,   568,
           340,   373,   645,  1049],
        [ 5975,   284,   502,   284,  3285,   326,    11,   287,   262,  6001,
           286,   465, 13476,    11,   339,   550,  5710,   465, 12036,    11,
          6405,   257,  5527, 27075]])


In [7]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cpu


In [8]:
model = GPT(GPT2_SMALL).to(device)

In [9]:
from evaluation.metrics import Metrics
metrics = Metrics()

In [10]:
trainer=Trainer(model,tokenizer,train_dataloader,val_dataloader,device,cross_entropy_loss,token_accuracy,CHECKPOINT_CONFIG,TRAIN_CONFIG,OPTIMIZER_CONFIG,SCHEDULER_CONFIG)

In [11]:
trainer.fit()

100%|██████████| 96/96 [00:09<00:00, 10.08it/s]


Checkpoint saved -> checkpoints\checkpoint_96.pt
Best checkpoint saved -> checkpoints\best_checkpoint.pt
Output text:
 Every effort moves you 153
after 1 epoch global step 96 the train loss 10.795750111341476 val loss 10.39136028289795 and train acc| 0.0 val acc| 0.0018939394503831863 


100%|██████████| 96/96 [00:10<00:00,  9.28it/s]


Checkpoint saved -> checkpoints\checkpoint_192.pt
Best checkpoint saved -> checkpoints\best_checkpoint.pt
Output text:
 Every effort moves you holding
after 2 epoch global step 192 the train loss 9.47642034292221 val loss 8.730690956115723 and train acc| 0.017361111531499773 val acc| 0.02083333395421505 


100%|██████████| 96/96 [00:10<00:00,  9.23it/s]


Checkpoint saved -> checkpoints\checkpoint_288.pt
Best checkpoint saved -> checkpoints\best_checkpoint.pt
Output text:
 Every effort moves you Miss
after 3 epoch global step 288 the train loss 7.864393929640452 val loss 7.549495220184326 and train acc| 0.03168402848920474 val acc| 0.024621212854981422 


100%|██████████| 96/96 [00:09<00:00,  9.95it/s]


Checkpoint saved -> checkpoints\checkpoint_384.pt
Best checkpoint saved -> checkpoints\best_checkpoint.pt
Output text:
 Every effort moves youGROUND
after 4 epoch global step 384 the train loss 6.867031628886859 val loss 7.002615451812744 and train acc| 0.03450520887660483 val acc| 0.039772726595401764 


100%|██████████| 96/96 [00:09<00:00,  9.78it/s]


Checkpoint saved -> checkpoints\checkpoint_480.pt
Best checkpoint saved -> checkpoints\best_checkpoint.pt
Output text:
 Every effort moves you painted
after 5 epoch global step 480 the train loss 6.427155862251918 val loss 6.807925224304199 and train acc| 0.038411458993020155 val acc| 0.0416666716337204 
